# Part A — Notebook 1: Data Download & Exploration
#
# **Research**: Beyond Gut Feel — AI vs Developer Intuition in Observability Decisions
#
# **Goal**: Download the AIOps 2018 KPI Anomaly Detection dataset,
# understand its structure, and visualize patterns.
#
# **Dataset**: Real production KPI time-series data from internet companies
# (Tsinghua NetMan Lab). 26 KPIs with binary anomaly labels.
#
# **Run time**: ~2 minutes
#
# ---

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import os

# Output directories
FIGURE_DIR = '../figures'
DATA_DIR = '../data'
os.makedirs(FIGURE_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

# Plot settings
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['figure.dpi'] = 100

# Reproducibility
np.random.seed(42)

print(f'Figures → {os.path.abspath(FIGURE_DIR)}')
print(f'Data   → {os.path.abspath(DATA_DIR)}')
print('Libraries loaded!')

## 2. Download the Dataset

The AIOps 2018 KPI dataset is hosted on GitHub by the Tsinghua NetMan lab.

It has two parts:
- **Training set** (`train.csv`): KPI data with anomaly labels — we learn patterns from this
- **Test set** (`test.csv` + `test_label.csv`): KPI data where we predict anomalies and check our accuracy

In [ ]:
# Download the dataset from the NetManAIOps GitHub repo
!git clone https://github.com/NetManAIOps/KPI-Anomaly-Detection.git /content/kpi-dataset

# Check what files we got
!ls -la /content/kpi-dataset/Preliminary_dataset/

In [ ]:
# Load the training data
train_df = pd.read_csv('/content/kpi-dataset/Preliminary_dataset/train.csv')

print(f'Dataset shape: {train_df.shape[0]:,} rows x {train_df.shape[1]} columns')
print(f'\nColumn names: {list(train_df.columns)}')
print(f'\nColumn data types:')
print(train_df.dtypes)
print(f'\nMemory usage: {train_df.memory_usage(deep=True).sum() / 1024 / 1024:.1f} MB')

## 3. First Look — What Does the Data Look Like?

Before doing anything fancy, always look at your raw data first.

**What to look for:**
- What are the columns?
- What do the values look like?
- Are there missing values?
- How many different KPIs are there?

In [ ]:
# Show the first 10 rows
# Think of each row as one Prometheus scrape result
train_df.head(10)

In [ ]:
# Show the last 10 rows
train_df.tail(10)

In [ ]:
# Basic statistics
# - count: how many non-null values
# - mean: average value
# - std: standard deviation (how spread out the values are)
# - min/max: range of values
train_df.describe()

In [ ]:
# Check for missing values
# Missing data is common in real metrics (network issues, restarts, etc.)
missing = train_df.isnull().sum()
print('Missing values per column:')
print(missing)
print(f'\nTotal missing: {missing.sum()}')

## 4. Understand the KPIs

The dataset contains multiple KPIs, each identified by a unique ID.
Each KPI is a different metric from a different service — like how your SDK creates
`apigateway_http_requests_total` vs `authentication_process_cpu_seconds_total`.

In [ ]:
# How many distinct KPIs?
kpi_ids = train_df['KPI ID'].unique()
print(f'Number of distinct KPIs: {len(kpi_ids)}')
print(f'\nKPI IDs:')
for i, kpi_id in enumerate(kpi_ids):
    kpi_data = train_df[train_df['KPI ID'] == kpi_id]
    anomaly_pct = kpi_data['label'].mean() * 100
    print(f'  KPI {i+1}: {kpi_id[:12]}... | {len(kpi_data):>8,} points | {anomaly_pct:.2f}% anomalous')

## 5. Class Balance — How Much is Anomalous?

This is **critical** for ML. If only 1% of data is anomalous (which is typical in monitoring),
a model that always says "normal" gets 99% accuracy but is completely useless.

This is exactly like your Prometheus alerts — most of the time everything is fine.
The challenge is catching the rare moments when it's not.

In [ ]:
# Overall class distribution
label_counts = train_df['label'].value_counts()
label_pcts = train_df['label'].value_counts(normalize=True) * 100

print('Overall class distribution:')
print(f'  Normal (0):    {label_counts[0]:>10,} ({label_pcts[0]:.2f}%)')
print(f'  Anomalous (1): {label_counts[1]:>10,} ({label_pcts[1]:.2f}%)')
print(f'  Ratio:         1 anomaly per {label_counts[0] // label_counts[1]} normal points')

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
colors = ['#2B5EA7', '#C25732']
axes[0].bar(['Normal', 'Anomalous'], label_counts.values, color=colors)
axes[0].set_title('Class Distribution (Count)')
axes[0].set_ylabel('Number of data points')
for i, v in enumerate(label_counts.values):
    axes[0].text(i, v + 1000, f'{v:,}', ha='center', fontweight='bold')

# Per-KPI anomaly percentage
kpi_anomaly_pcts = train_df.groupby('KPI ID')['label'].mean() * 100
kpi_anomaly_pcts = kpi_anomaly_pcts.sort_values(ascending=True)
axes[1].barh(range(len(kpi_anomaly_pcts)), kpi_anomaly_pcts.values, color='#C25732')
axes[1].set_title('Anomaly % per KPI')
axes[1].set_xlabel('Anomaly percentage (%)')
axes[1].set_ylabel('KPI index')
axes[1].set_yticks(range(len(kpi_anomaly_pcts)))
axes[1].set_yticklabels([f'KPI {i+1}' for i in range(len(kpi_anomaly_pcts))], fontsize=8)

plt.tight_layout()
plt.show()

## 6. Visualize the Time Series

Now let's see what these KPIs actually look like over time.
This is like opening a Grafana dashboard and looking at the graphs.

Red dots = anomalous points. The ML model needs to learn what makes those red dots different.

In [ ]:
# Convert timestamp to readable datetime
train_df['datetime'] = pd.to_datetime(train_df['timestamp'], unit='s')

# Plot first 4 KPIs
fig, axes = plt.subplots(4, 1, figsize=(16, 16))

for idx, kpi_id in enumerate(kpi_ids[:4]):
    kpi_data = train_df[train_df['KPI ID'] == kpi_id].copy()
    normal = kpi_data[kpi_data['label'] == 0]
    anomalous = kpi_data[kpi_data['label'] == 1]

    axes[idx].plot(normal['datetime'], normal['value'],
                   '.', markersize=1, color='#2B5EA7', alpha=0.5, label='Normal')
    axes[idx].plot(anomalous['datetime'], anomalous['value'],
                   '.', markersize=3, color='#C25732', alpha=0.8, label='Anomalous')
    axes[idx].set_title(f'KPI {idx+1} ({kpi_id[:12]}...)', fontweight='bold')
    axes[idx].set_ylabel('Value')
    axes[idx].legend(loc='upper right', markerscale=5)

axes[-1].set_xlabel('Time')
plt.suptitle('KPI Time Series — Normal (blue) vs Anomalous (red)', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 7. Zoom Into Anomalies

Let's zoom into one KPI and look at what anomalies look like up close.
Understanding the **shape** of anomalies helps us pick the right ML approach.

In [ ]:
# Pick one KPI and zoom into a window around an anomaly
sample_kpi_id = kpi_ids[0]
kpi_data = train_df[train_df['KPI ID'] == sample_kpi_id].copy()
kpi_data = kpi_data.sort_values('timestamp').reset_index(drop=True)

# Find the first anomaly window
anomaly_indices = kpi_data[kpi_data['label'] == 1].index
if len(anomaly_indices) > 0:
    first_anomaly_idx = anomaly_indices[0]
    # Take 500 points before and after
    start = max(0, first_anomaly_idx - 500)
    end = min(len(kpi_data), first_anomaly_idx + 500)
    window = kpi_data.iloc[start:end]

    normal_w = window[window['label'] == 0]
    anomalous_w = window[window['label'] == 1]

    fig, ax = plt.subplots(figsize=(16, 5))
    ax.plot(normal_w['datetime'], normal_w['value'], '-', color='#2B5EA7', alpha=0.7, label='Normal')
    ax.scatter(anomalous_w['datetime'], anomalous_w['value'],
               color='#C25732', s=20, zorder=5, label='Anomalous')
    ax.set_title(f'Zoomed View — KPI 1 Around First Anomaly', fontweight='bold')
    ax.set_xlabel('Time')
    ax.set_ylabel('Value')
    ax.legend()
    ax.axvspan(anomalous_w['datetime'].min(), anomalous_w['datetime'].max(),
               alpha=0.1, color='#C25732', label='Anomaly window')
    plt.tight_layout()
    plt.show()

    print(f'Anomaly window duration: {anomalous_w["datetime"].max() - anomalous_w["datetime"].min()}')
    print(f'Points in window: {len(window)} ({len(anomalous_w)} anomalous)')

## 8. Statistical Summary per KPI

Let's compare normal vs anomalous data points statistically.
If anomalous points have very different statistical properties,
even simple models might catch them.

In [ ]:
# Compare statistics: normal vs anomalous for each KPI
summary_rows = []

for i, kpi_id in enumerate(kpi_ids):
    kpi_data = train_df[train_df['KPI ID'] == kpi_id]
    normal = kpi_data[kpi_data['label'] == 0]['value']
    anomalous = kpi_data[kpi_data['label'] == 1]['value']

    summary_rows.append({
        'KPI': f'KPI {i+1}',
        'Normal Mean': f'{normal.mean():.2f}',
        'Normal Std': f'{normal.std():.2f}',
        'Anomaly Mean': f'{anomalous.mean():.2f}',
        'Anomaly Std': f'{anomalous.std():.2f}',
        'Mean Diff (%)': f'{abs(anomalous.mean() - normal.mean()) / (normal.std() + 1e-8) * 100:.1f}%',
        'Anomaly %': f'{len(anomalous) / len(kpi_data) * 100:.2f}%'
    })

summary_df = pd.DataFrame(summary_rows)
print('Normal vs Anomalous — Statistical Comparison')
print('=' * 90)
print(summary_df.to_string(index=False))

## 9. Scrape Interval Analysis

Like Prometheus, these KPIs are scraped at regular intervals.
Let's verify the scrape interval — this tells us the time resolution of our data.

In [ ]:
# Check scrape interval for each KPI
for i, kpi_id in enumerate(kpi_ids[:6]):
    kpi_data = train_df[train_df['KPI ID'] == kpi_id].sort_values('timestamp')
    intervals = kpi_data['timestamp'].diff().dropna()
    common_interval = intervals.mode()[0]
    print(f'KPI {i+1}: interval = {int(common_interval)}s ({int(common_interval/60)}min) | '
          f'time span = {(kpi_data["timestamp"].max() - kpi_data["timestamp"].min()) / 86400:.1f} days')

## Summary
#
# Key findings from this exploration:
# - 26 KPIs, ~2.16% anomalous overall (highly imbalanced)
# - Anomaly rates vary widely across KPIs (some <1%, some >5%)
# - 1-minute scrape interval
# - Anomalies manifest as pattern changes, not just simple spikes
# - No missing values in the dataset
#
# **Next**: `02_baseline_and_features.ipynb` — Feature engineering & threshold baseline